In [ ]:
from langchain_community.vectorstores import Chroma
from app.core.config import get_db_path, get_env
from app.embeddings.huggingface_tei import HuggingFaceTEIEmbedder

tei_url = get_env("TEI_URL", "http://localhost:8080") or "http://localhost:8080"
embedder = HuggingFaceTEIEmbedder(base_url=tei_url)

db_path = get_db_path().as_posix()
vectorstore = Chroma(
    persist_directory=db_path,
    embedding_function=embedder,
    collection_name="notebook_test",
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})


C:\Users\Tien\AppData\Local\Temp\ipykernel_14964\3734833608.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [4]:
from langchain_community.chat_models import ChatOllama

llm = ChatOllama(
    model="gemma3:1b",
    base_url="http://localhost:11434",
    temperature=0.2,
)


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
C:\Users\Tien\AppData\Local\Temp\ipykernel_14964\3925294126.py:3: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(


In [5]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an AI assistant that answers questions using only the provided context.
If the answer cannot be found in the context, say "I don't know".

Context:
{context}

Question:
{question}

Answer:
""")


In [6]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "context": retriever | (lambda docs: "\n\n".join(d.page_content for d in docs)),
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
)


In [8]:
response = rag_chain.invoke("What steps did the author include as the learning path?")
print(response.content)


1.  Learn how to use the frameworks like Lang Chain, Langraph, and Pidentici.
2.  Study the core building blocks of what makes up an LLM-based system: deterministic logic with LLMs, inputs, prompts, context windows, output, feedback loops.
3.  Sketch cognitive architectures, which means creating simple block diagrams to show how data flows through your system and where you can strategically place large language models.
4.  Learn how to create unit tests, human-annotated data sets, and LLM as judges for evaluations.
5.  Learn how to store data sets for regression tests.
6.  Understand the concept of guardrails for safety and security.
7.  Start with OpenAI and work with large language models.
8.  Learn about asynchronous programming and background workers.
9.  Learn how to capture data sets, store them, and run these experiments.
10. Understand the concept of prompt engineering.

